# VORTEX observer and the LPIPS-optimised control, n=100 (`tab:app-realism`)

Printed output only -- no file in the archive holds `tab:app-realism`'s numbers.

**Needs:** DTD, the CNT checkpoint, cached VORTEX observer weights/config.

**Maintained runnable path:** none. This notebook is the only record of `tab:app-realism`.

In [ ]:
# --- installs ---
!pip install -q pot geomloss lpips timm==1.0.11 einops
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'   # ~2-5 min

import os, sys, torch, torch.nn.functional as F, numpy as np
from pathlib import Path
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- CNT ---
os.chdir('/content')
!rm -rf cnt-siga24
!git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24
!pip install -q -r cnt-siga24/requirements.in

from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT    = f'{PROJECT_DIR}/dtd/images'
!rm -rf /content/cnt-siga24/ckpts
!ln -sfn "{PROJECT_DIR}/cnt_ckpts" /content/cnt-siga24/ckpts

os.chdir('/content/cnt-siga24'); sys.path.insert(0, '/content/cnt-siga24')
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"), device=device)
print("CNT loaded")

In [ ]:
# ============================================================
# Cell 2 — CNT interpolators + lpips_opt control
# ============================================================
IMG_SIZE = 256
FMS = [round(IMG_SIZE * r) for r in model.generator.feature_map_size_ratios]

def load_texture(path):
    return load_image(Path(path), IMG_SIZE).to(device)

def get_generator_inputs(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor)
        layout = model.splat(blob)
        bf = layout["layer_level_layouts"][-1][0]["blob_features"].clone()
        si = layout["layer_level_layouts"][-1][0]["score_img"].clone()
    return blob, bf, si

def decode_interp(blob_ref, bf_new, si_new):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf_new
        layout["layer_level_layouts"][-1][0]["score_img"] = si_new
        return model.decode(layout)

def build_style_maps(bf, si, fms):
    sp = {}; s = si; sp[s.shape[-1]] = s
    while s.shape[-1] > min(fms):
        s = F.interpolate(s, s.shape[-1]//2, mode="bilinear", align_corners=False); sp[s.shape[-1]] = s
    return {size: torch.einsum("bnc,bnhw->bchw", bf, sp[size]) for size in set(fms)}

def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        si = layout["layer_level_layouts"][-1][0]["score_img"]
        ratios = model.generator.feature_map_size_ratios
        fms = [round(si.shape[-1]*r) for r in ratios]
        sp = {}; s = si; sp[s.shape[-1]] = s
        while s.shape[-1] > min(fms):
            s = F.interpolate(s, s.shape[-1]//2, mode="bilinear", align_corners=False); sp[s.shape[-1]] = s
        feat_in = sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1, model.generator.n_embedding,1,1)
        x = feat_in.clone()
        for i in range(len(fms)):
            x = model.generator.spade_blocks[i](x, sm[fms[i]])
            if i == len(fms)-1: break
            elif fms[i] != fms[i+1]:
                x = F.interpolate(x, size=x.shape[-1]*2, mode="bilinear", align_corners=False)
        return torch.nn.functional.tanh(x.unsqueeze(1)).flatten(1,2)

def sinkhorn_transport(src, tgt, eps=0.05, n_iters=100):
    N, M = src.shape[0], tgt.shape[0]
    C = torch.cdist(src, tgt).pow(2); K = torch.exp(-C/eps)
    a = torch.ones(N,1,device=src.device)/N; b = torch.ones(M,1,device=src.device)/M
    u = torch.ones(N,1,device=src.device)
    for _ in range(n_iters):
        v = b/(K.T@u+1e-10); u = a/(K@v+1e-10)
    plan = u*K*v.T; plan = plan/(plan.sum(1,keepdim=True)+1e-10)
    return plan @ tgt

def build_d(path_a, path_b):
    ia, ib = load_texture(path_a), load_texture(path_b)
    ba, bfa, sia = get_generator_inputs(ia)
    bb, bfb, sib = get_generator_inputs(ib)
    return {"blob_a_gen":ba, "bf_a":bfa, "si_a":sia,
            "blob_b_gen":bb, "bf_b":bfb, "si_b":sib, "ia":ia, "ib":ib}

def interpolate_sinkhorn_pixel(d, eta, n_subsample=4096):        # OT (ours)
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS); source, target = sm_a[largest][0], sm_b[largest][0]
    C_ch, H, W = source.shape
    sf = source.permute(1,2,0).reshape(-1,C_ch); tf = target.permute(1,2,0).reshape(-1,C_ch)
    N = sf.shape[0]
    idx_s = torch.randperm(N,device=device)[:n_subsample]; idx_t = torch.randperm(N,device=device)[:n_subsample]
    with torch.no_grad(): tt = sinkhorn_transport(sf[idx_s], tf[idx_t])
    nn = torch.cdist(sf, sf[idx_s]).argmin(1); all_t = tt[nn]
    interp = ((1-eta)*sf + eta*all_t).reshape(H,W,C_ch).permute(2,0,1).unsqueeze(0)
    sm = {size:(interp if size==largest else F.interpolate(interp,size=size,mode="bilinear",align_corners=False)) for size in set(FMS)}
    return decode_with_style_maps(d["blob_a_gen"], sm)

def interpolate_linear_pixel(d, eta):                            # pixel_linear
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    sm = {size:(1-eta)*sm_a[size]+eta*sm_b[size] for size in set(FMS)}
    return decode_with_style_maps(d["blob_a_gen"], sm)

def interpolate_linear_texton(d, eta):                           # texton_linear
    bf = (1-eta)*d["bf_a"] + eta*d["bf_b"]
    si = (1-eta)*d["si_a"] + eta*d["si_b"]
    return decode_interp(d["blob_a_gen"], bf, si)

def to_pil(decoded):
    a = ((decoded.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))

# --- lpips_opt negative control (from-noise; produces genuine static) ---
import lpips as lpips_lib
lpips_fn = lpips_lib.LPIPS(net='vgg').to(device).eval()
for p in lpips_fn.parameters(): p.requires_grad_(False)

def lpips_opt_midpoint_v2(pil_a, pil_b, steps=800, lr=0.05):
    def prep(pil):
        x = torch.from_numpy(np.asarray(pil.convert('RGB').resize((256,256))).copy()).float()
        return (x/127.5 - 1).permute(2,0,1).unsqueeze(0).to(device)
    A, B = prep(pil_a), prep(pil_b)
    M = (torch.randn_like(A)*0.5).clamp(-1,1).clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([M], lr=lr)
    for _ in range(steps):
        dA, dB = lpips_fn(M, A), lpips_fn(M, B)
        loss = (dA-dB).pow(2).mean() + 0.02*(dA+dB).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): M.clamp_(-1,1)
    arr = ((M.detach().squeeze(0).clamp(-1,1)+1)/2).permute(1,2,0).cpu().numpy()
    return Image.fromarray((arr*255).astype(np.uint8))

print("interpolators + control ready:",
      [f for f in ['build_d','interpolate_sinkhorn_pixel','interpolate_linear_pixel',
                   'interpolate_linear_texton','lpips_opt_midpoint_v2'] if f in dir()])

In [ ]:
!git clone -q https://github.com/scabini/VORTEX.git /content/VORTEX
os.chdir('/content/VORTEX'); sys.path.insert(0, '/content/VORTEX')
from models import VORTEX
from torchvision import transforms

vortex_model = VORTEX('beitv2_base_patch16_224.in1k_ft_in22k_in1k', 224)
vortex_model.to(device)          # in place — do NOT reassign (their .to/.eval return None)
vortex_model.eval()
for p in vortex_model.parameters(): p.requires_grad_(False)

vtf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    transforms.Resize((224,224)),
])
def vortex_feats(pil):
    x = vtf(pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad(): f = vortex_model(x)
    return f.detach().cpu().numpy().ravel()

# smoke test
a, b = Image.effect_noise((256,256),60).convert('RGB'), Image.effect_noise((256,256),20).convert('RGB')
fa1, fa2, fb = vortex_feats(a), vortex_feats(a), vortex_feats(b)
print(f"VORTEX dim {fa1.shape}  self-dist {np.linalg.norm(fa1-fa2):.2e}  A-vs-B {np.linalg.norm(fa1-fb):.3f}")

In [ ]:
import glob, random, json, time
from scipy.stats import wilcoxon
random.seed(1)
os.chdir('/content/cnt-siga24')   # CNT needs its dir for generation

cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(cat, seed):
    return random.Random(seed).choice(sorted(glob.glob(f'{DTD_ROOT}/{cat}/*.jpg')))

N_PAIRS = 100
pairs = []
while len(pairs) < N_PAIRS:
    x,y = random.sample(cats,2); pairs.append((dtd_path(x,len(pairs)), dtd_path(y,len(pairs)+7000)))
HELD = [dtd_path(random.choice(cats), 30000+i) for i in range(120)]

held_v = np.stack([vortex_feats(Image.open(p)) for p in HELD])
held_v = held_v[~np.isnan(held_v).any(axis=1)]
vmu, vsd = held_v.mean(0), held_v.std(0); vsd[vsd<1e-6]=1.0
held_vz = (held_v-vmu)/vsd
def vz(f): return (f-vmu)/vsd
def vdist(u,v): return np.linalg.norm(vz(u)-vz(v))
print(f"{N_PAIRS} pairs, {len(held_v)} held reals")

METHODS = ['pixel_linear','texton_linear','ot','lpips_opt']
VCACHE = {}; RV = {m:{'balance':[],'proximity':[],'structure':[]} for m in METHODS}
t0=time.time()
for k,(pa,pb) in enumerate(pairs):
    try:
        d = build_d(pa,pb); pil_a,pil_b = Image.open(pa),Image.open(pb)
        mids = {'pixel_linear':to_pil(interpolate_linear_pixel(d,0.5)),
                'texton_linear':to_pil(interpolate_linear_texton(d,0.5)),
                'ot':to_pil(interpolate_sinkhorn_pixel(d,0.5)),
                'lpips_opt':lpips_opt_midpoint_v2(pil_a,pil_b)}
    except Exception as e:
        print(f"pair {k}: {type(e).__name__}"); continue
    fA,fB = vortex_feats(pil_a), vortex_feats(pil_b)
    entry = {'A':fA.tolist(),'B':fB.tolist()}
    for m in METHODS: entry[m]=vortex_feats(mids[m]).tolist()
    VCACHE[k]=entry
    dAB = vdist(fA,fB)+1e-9
    for m in METHODS:
        fM=np.array(entry[m]); dMA,dMB=vdist(fM,fA),vdist(fM,fB)
        RV[m]['balance'].append(abs(dMA-dMB)/dAB)
        RV[m]['proximity'].append(float(np.min(np.linalg.norm(held_vz-vz(fM),axis=1))))
        RV[m]['structure'].append(0.5*(dMA+dMB))
    if k%10==0: print(f"  {k+1}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)")

json.dump(VCACHE, open(f'{PROJECT_DIR}/vortex_cache_n{N_PAIRS}.json','w'))
def mci(x): x=np.array(x); return (x.mean(),1.96*x.std()/max(len(x),1)**0.5)
print(f"\nVORTEX n={N_PAIRS}\n{'method':14}{'balance↓':>14}{'proximity↓':>15}{'structure↓':>15}")
for m in METHODS:
    (b,bc),(p,pc),(s,sc)=mci(RV[m]['balance']),mci(RV[m]['proximity']),mci(RV[m]['structure'])
    print(f"{m:14}{b:8.3f}±{bc:.3f}{p:10.2f}±{pc:.2f}{s:10.2f}±{sc:.2f}")
for base in ['pixel_linear','texton_linear','lpips_opt']:
    print(f"\n  vs {base}:")
    for meas in ['balance','proximity','structure']:
        o,bs=np.array(RV['ot'][meas]),np.array(RV[base][meas]); n=min(len(o),len(bs))
        w=int((o[:n]<bs[:n]).sum())
        try: pv=wilcoxon(o[:n],bs[:n],alternative='less').pvalue
        except: pv=float('nan')
        print(f"    {meas:10}: {w:3d}/{n}  p={pv:.2e}  ({o[:n].mean():.3f} vs {bs[:n].mean():.3f})  {'OT wins' if w>n/2 else 'OT loses'}")

In [ ]:
# ============================================================
# Kynkäänniemi realism score for midpoints (NeurIPS 2019, Eq. 3)
# Measures how far onto the REAL-texture manifold each method's midpoint lands.
# Higher = more realistic / on-manifold. This is the accepted interpolation metric.
# Requires: inc_feats (real InceptionV3 -> 2048-d) from your n=1000 setup,
#           the 4 interpolators, to_pil, DTD_ROOT, held-out real DTD images.
# ============================================================
import numpy as np, glob, random, torch
from PIL import Image

# --- if inc_feats isn't in this kernel, (re)build the InceptionV3 extractor ---
if 'inc_feats' not in dir():
    from torchvision.models import inception_v3, Inception_V3_Weights
    import torch.nn.functional as F
    _inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
    _inc.fc = torch.nn.Identity()
    for p in _inc.parameters(): p.requires_grad_(False)
    _M = torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
    _S = torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
    @torch.no_grad()
    def inc_feats(pils, bs=32):
        out=[]
        for i in range(0,len(pils),bs):
            xs=[]
            for pil in pils[i:i+bs]:
                a=np.asarray(pil.convert('RGB').resize((256,256))).copy()
                xs.append(torch.from_numpy(a).float().permute(2,0,1)/255.)
            b=torch.stack(xs).to(device)
            b=F.interpolate(b,size=299,mode='bilinear',align_corners=False)
            b=(b-_M)/_S
            out.append(_inc(b).cpu().numpy())
        return np.concatenate(out)

# --- build the REAL manifold from held-out DTD textures (Inception features) ---
random.seed(2)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(cat, seed): return random.Random(seed).choice(sorted(glob.glob(f'{DTD_ROOT}/{cat}/*.jpg')))
REAL_IMGS = [Image.open(dtd_path(random.choice(cats), 50000+i)) for i in range(500)]  # real manifold
Phi_r = inc_feats(REAL_IMGS)
print("real manifold:", Phi_r.shape)

# --- k-NN radii of the real manifold, with the median-pruning from the paper ---
K = 3
D_rr = np.linalg.norm(Phi_r[:,None]-Phi_r[None,:], axis=2)
np.fill_diagonal(D_rr, np.inf)
radii = np.sort(D_rr, axis=1)[:, K-1]              # distance to k-th NN for each real point
keep = radii <= np.median(radii)                    # PAPER: discard half with largest radii
Phi_r_pruned, radii_pruned = Phi_r[keep], radii[keep]
print(f"kept {keep.sum()}/{len(Phi_r)} real points after median-radius pruning")

def realism_score(phi_g):
    # Eq.3: max over real points of  radius(phi_r) / ||phi_g - phi_r||
    d = np.linalg.norm(Phi_r_pruned - phi_g, axis=1) + 1e-9
    return float(np.max(radii_pruned / d))

# --- score the 4 methods' midpoints on the SAME pairs as before ---
random.seed(1)
N = 100
pairs = []
while len(pairs) < N:
    x,y = random.sample(cats,2); pairs.append((dtd_path(x,len(pairs)), dtd_path(y,len(pairs)+7000)))

METHODS = ['pixel_linear','texton_linear','ot','lpips_opt']
mid_pils = {m:[] for m in METHODS}
for k,(pa,pb) in enumerate(pairs):
    d = build_d(pa,pb); pil_a,pil_b = Image.open(pa),Image.open(pb)
    mid_pils['pixel_linear'].append(to_pil(interpolate_linear_pixel(d,0.5)))
    mid_pils['texton_linear'].append(to_pil(interpolate_linear_texton(d,0.5)))
    mid_pils['ot'].append(to_pil(interpolate_sinkhorn_pixel(d,0.5)))
    mid_pils['lpips_opt'].append(lpips_opt_midpoint_v2(pil_a,pil_b))
    if k%20==0: print(f"  gen {k+1}/{N}")

from scipy.stats import wilcoxon
scores = {}
for m in METHODS:
    feats = inc_feats(mid_pils[m])
    scores[m] = np.array([realism_score(f) for f in feats])

print(f"\nRealism score (higher = more on real-texture manifold; R>=1 means inside)")
print(f"{'method':14}{'mean R':>10}{'median R':>10}{'% R>=1':>9}")
for m in METHODS:
    s=scores[m]; print(f"{m:14}{s.mean():10.3f}{np.median(s):10.3f}{100*(s>=1).mean():8.0f}%")

print("\nOT vs baselines (higher R = better; OT should WIN = be higher):")
for base in ['pixel_linear','texton_linear','lpips_opt']:
    o,b = scores['ot'], scores[base]
    wins = int((o>b).sum())
    pv = wilcoxon(o,b,alternative='greater').pvalue
    print(f"  vs {base:14}: OT wins {wins}/{len(o)}  p={pv:.2e}  ({o.mean():.3f} vs {b.mean():.3f})  {'OT wins' if wins>len(o)/2 else 'OT loses'}")

In [ ]:
import os
import glob, json, os
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
# look for saved ratio / LPIPS result files
for kw in ['ratio','lpips','ptd','dtd']:
    hits = glob.glob(f'{BASE}/**/*{kw}*', recursive=True)
    jsons = [h for h in hits if h.endswith('.json')]
    for j in jsons:
        print(j.replace(BASE+'/',''))

In [ ]:
import os
import glob, json, os
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
FOLD = f'{BASE}/interpolation_systematic/ptd_large_benchmark'

# list EVERYTHING in that benchmark folder
print("=== all files in ptd_large_benchmark ===")
for f in sorted(glob.glob(f'{FOLD}/**/*', recursive=True)):
    if os.path.isfile(f):
        sz = os.path.getsize(f)/1e6
        print(f"  {f.replace(FOLD+'/',''):60s} {sz:.2f} MB")